In [9]:
import os
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
con = duckdb.connect()

def load_queries(path):
    text = Path(path).read_text(encoding="utf-8")
    queries, current_name, current_sql = {}, None, []
    for line in text.splitlines():
        if line.strip().startswith("-- name:"):
            if current_name is not None:
                queries[current_name] = "\n".join(current_sql).strip()
            current_name = line.split("-- name:", 1)[1].strip()
            current_sql = []
        else:
            current_sql.append(line)
    if current_name is not None:
        queries[current_name] = "\n".join(current_sql).strip()
    return queries

def run(queries, name):
    return con.execute(queries[name]).df()

In [10]:
for f in ["sql/00_setup.sql", "sql/01_scope.sql", "sql/01_clean.sql", "sql/02_features.sql"]:
    for name, sql in load_queries(f).items():
        if name.startswith("create_"):
            con.execute(sql)

feat = load_queries("sql/02_features.sql")
display(run(feat, "features_reconcile").T)

,0
n_rows,20405666
pu_zone_null,0
pu_borough_null,0
min_hour,0
max_hour,23
min_isodow,1
max_isodow,7
n_pickup_zones,262
min_request_date,2024-12-31 00:00:00
max_request_date,2025-02-01 00:00:00


In [16]:
import pandas as pd
from pathlib import Path

a = load_queries("sql/02a_wait_profile.sql")
prof = run(a, "wait_profile_by_company")
display(prof)

expected = {"HV0003": (15356455, 15235488, 120967),
            "HV0005": (5049211, 4970390, 78821),
            "ALL": (20405666, 20205878, 199788)}
for _, r in prof.iterrows():
    exp = expected[r["hvfhs_license_num"]]
    got = (int(r["n_scoped"]), int(r["n_valid"]), int(r["n_excluded_negative_wait"]))
    print(r["company"], "OK" if got == exp else f"MISMATCH {got} vs {exp}")
p = prof.set_index("company")

cols = [
    "mean_min", "p50_min", "p75_min",
    "p90_min", "p95_min", "p99_min"
]

lyft = pd.to_numeric(
    p.loc["Lyft", cols], errors="coerce"
).to_numpy(dtype="float64")

uber = pd.to_numeric(
    p.loc["Uber", cols], errors="coerce"
).to_numpy(dtype="float64")

gap = pd.Series(
    (lyft - uber) * 60,
    index=cols
).round(1)

display(
    gap.rename("Lyft minus Uber (seconds)").to_frame()
)

,hvfhs_license_num,company,n_scoped,n_valid,n_excluded_negative_wait,pct_excluded,mean_min,p50_min,p50_sec,p75_min,p90_min,p95_min,p99_min
0,HV0003,Uber,15356455,15235488,120967,0.788,4.556,3.800,228.0,5.517,8.033,10.217,15.783
1,HV0005,Lyft,5049211,4970390,78821,1.561,4.691,4.083,245.0,5.767,8.033,9.867,14.533
2,ALL,All,20405666,20205878,199788,0.979,4.589,3.867,232.0,5.583,8.033,10.117,15.500


Uber OK
Lyft OK
All OK


,Lyft minus Uber (seconds)
mean_min,8.1
p50_min,17.0
p75_min,15.0
p90_min,0.0
p95_min,-21.0
p99_min,-75.0


In [15]:
bins = run(a, "wait_bins_by_company")
display(bins)

for comp, tot in [("HV0003", 15356455), ("HV0005", 5049211)]:
    b = bins[bins.hvfhs_license_num == comp]
    print(comp, "bins sum to n_scoped:", int(b["n_rows"].sum()) == tot,
          "| valid bins sum:", int(b.loc[b.bin_start_min >= 0, "n_rows"].sum()))

def share(comp, lo, hi):
    m = bins[(bins.hvfhs_license_num == comp) & (bins.bin_start_min >= lo) & (bins.bin_start_min < hi)]
    return round(float(m["pct_of_valid"].sum()), 2)

for comp in ["HV0003", "HV0005"]:
    print(comp, "2-<5 min:", share(comp, 2, 5), "| 5-<10 min:", share(comp, 5, 10))

out = Path("dashboard/data"); out.mkdir(parents=True, exist_ok=True)
prof.to_csv(out / "2a_wait_profile_by_company.csv", index=False)
bins.to_csv(out / "2a_wait_bins_by_company.csv", index=False)

,hvfhs_license_num,company,bin_start_min,bin_label,n_rows,pct_of_scoped,pct_of_valid,cum_pct_of_valid
0,HV0003,Uber,-1,excluded: negative wait,120967,0.788,NaN,NaN
1,HV0003,Uber,0,0-<1 min,208656,1.359,1.370,1.370
2,HV0003,Uber,1,1-<2 min,1546422,10.070,10.150,11.520
3,HV0003,Uber,2,2-<3 min,3168426,20.633,20.796,32.316
4,HV0003,Uber,3,3-<4 min,3262628,21.246,21.415,53.731
5,HV0003,Uber,4,4-<5 min,2354805,15.334,15.456,69.187
6,HV0003,Uber,5,5-<6 min,1528555,9.954,10.033,79.220
7,HV0003,Uber,6,6-<7 min,977073,6.363,6.413,85.633
8,HV0003,Uber,7,7-<8 min,639972,4.167,4.201,89.833
9,HV0003,Uber,8,8-<9 min,432905,2.819,2.841,92.675


HV0003 bins sum to n_scoped: True | valid bins sum: 15235488
HV0005 bins sum to n_scoped: True | valid bins sum: 4970390
HV0003 2-<5 min: 57.67 | 5-<10 min: 25.46
HV0005 2-<5 min: 55.58 | 5-<10 min: 29.81
